In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"            

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('ggplot')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (12, 8),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np

import datetime
import pytz
NY_tz = pytz.timezone("America/New_York") 
CHI_tz = pytz.timezone("America/Chicago") 
UTC_tz = pytz.timezone("UTC") 

In [2]:
import rateslib as rl
import QuantLib as ql

from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP

from Query.IRSwaps.IRSwapQuery import IRSwapQuery
from Query.IRSwaps.IRSwapStructure import IRSwapStructureFunctionMap, IRSwapStructure
from Query.IRSwaps.IRSwapValue import IRSwapValueFunctionMap, IRSwapValue

from MDP.IRSwaps.SDR_INTRADAY.rl_curve_utils.stir_curve_building_utils import cme_code_effective_date, first_business_day_next_month
from Query.IRSwaps.backends.quantlib.utils import ql_date_to_datetime, datetime_to_ql_date

In [3]:
stir_curve_mdp, curve = IRSwapsMDP(source="SDR_INTRADAY-rl_usd_ois_stir_q12x12"), "USD-FEDFUNDS"
# stir_curve_mdp, curve = IRSwapsMDP(source="SDR_INTRADAY-rl_usd_sofr_stir_q13x10"), "USD-SOFR-1D"

In [4]:
ts = "live"
# ts = NY_tz.localize(datetime.datetime(2025, 9, 24, 17, 00))
# live_curve = stir_curve_mdp._get_curve(curve_name=curve, timestamp=ts, kwargs={"force_refresh": True})
live_curve = stir_curve_mdp._get_curve(curve_name=curve, timestamp=ts, kwargs={"force_refresh": False})

SUCCESS: `conv_tol` reached after 4 iterations (levenberg_marquardt), `f_val`: 0.0006051174401727932, `time`: 0.1398s
SUCCESS: `conv_tol` reached after 4 iterations (levenberg_marquardt), `f_val`: 48135.24114911048, `time`: 0.1168s


In [5]:
fomc_dated_ois = {
    # "sep25": (rl.dt(2025, 9, 17), rl.dt(2025, 10, 29)),
    "oct25": (rl.dt(2025, 10, 29), rl.dt(2025, 12, 10)),
    "dec25": (rl.dt(2025, 12, 10), rl.dt(2026, 1, 28)),
    "jan26": (rl.dt(2026, 1, 28), rl.dt(2026, 3, 18)),
    "mar26": (rl.dt(2026, 3, 18), rl.dt(2026, 4, 29)),
    "apr26": (rl.dt(2026, 4, 29), rl.dt(2026, 6, 17)),
    "jun26": (rl.dt(2026, 6, 17), rl.dt(2026, 7, 29)),
    "jul26": (rl.dt(2026, 7, 29), rl.dt(2026, 9, 16)),
    "sep26": (rl.dt(2026, 9, 16), rl.dt(2026, 10, 28)),
    "oct26": (rl.dt(2026, 10, 28), rl.dt(2026, 12, 9)),
    "dec26": (rl.dt(2026, 12, 9), rl.dt(2027, 1, 27)),
}


def rl_fomc_swap(meeting: str, risk=None, notional=None, rl_irs_swap_kargs={}):
    assert risk is not None or notional is not None, "Must pass in `risk` or `notional`"

    if risk is not None:
        unit_delta = rl.IRS(
            effective=fomc_dated_ois[meeting][0], termination=fomc_dated_ois[meeting][1], spec="usd_irs_lt_2y", curves=live_curve.handle(), notional=1
        ).analytic_delta(live_curve.handle())
        notional = risk / unit_delta

    fomc_swap = rl.IRS(
        effective=fomc_dated_ois[meeting][0],
        termination=fomc_dated_ois[meeting][1],
        spec="usd_irs_lt_2y",
        curves=live_curve.handle(),
        notional=notional,
        **rl_irs_swap_kargs
    )
    return fomc_swap


meeting = "oct25"
rl_fomc_swap(meeting=meeting, risk=1).rate().real.__round__(3)

3.889

In [6]:
mid = 4.08
for m in fomc_dated_ois.keys():
    fomc_swap = rl_fomc_swap(meeting=m, risk=1)
    rate = fomc_swap.rate().real
    print(f"{m}, {rate:.3f}, {(mid - rate) * 100:.3f}")

oct25, 3.889, 19.070
dec25, 3.718, 36.162
jan26, 3.620, 45.970
mar26, 3.525, 55.518
apr26, 3.471, 60.908
jun26, 3.334, 74.580
jul26, 3.257, 82.298
sep26, 3.219, 86.095
oct26, 3.182, 89.810
dec26, 3.137, 94.268


# Risk:

In [7]:
def rl_sfr(imm: str, risk=None, contracts=None):
    assert risk or contracts, "must pass in risk or contracts"
    if not contracts:
        contracts = -int(risk / 25)
    return rl.STIRFuture(
        effective=rl.scheduling.get_imm(code=imm),
        termination=rl.scheduling.next_imm(rl.scheduling.get_imm(code=imm)),
        spec="usd_stir",
        curves=live_curve.handle(),
        contracts=contracts,
    )


def rl_ser(code: str, risk=None, contracts=None):
    assert risk or contracts, "must pass in risk or contracts"
    if not contracts:
        contracts = -int(risk / 41.67)

    eff = cme_code_effective_date(code)
    
    return rl.STIRFuture(
        effective=eff,
        # termination=ql_date_to_datetime(ql.UnitedStates(ql.UnitedStates.SOFR).endOfMonth(datetime_to_ql_date(eff))),
        termination=first_business_day_next_month(eff),
        spec="usd_stir1",
        curves=live_curve.handle(),
        roll="som",
        contracts=contracts, 
        leg2_fixings=live_curve.index(),
    )


irssfm = IRSwapStructureFunctionMap(curve=live_curve.handle())


def rl_irsq(q: IRSwapQuery):
    pkg, _ = irssfm.apply(q.structure, tenor=q.tenor, **q.structure_kwargs)
    return rl.Portfolio(pkg)

In [8]:
contracts = ["U25", "V25", "X25", "Z25", "F26", "G26", "H26", "J26", "K26", "M26", "N26", "Q26", "U26"]
stir1 = {}
for c in contracts:
    stir1[c] = rl_ser(code=c, contracts=1) 

fomcs = {}
for f in fomc_dated_ois.keys():
    fomcs[f] = rl_fomc_swap(meeting=f, risk=1) 


# rl_risk_instruments = stir1 | fomcs
rl_risk_instruments = stir1 
# rl_risk_instruments = fomcs 
rl_risk_solver = rl.Solver(
    curves=[live_curve.handle()],
    instruments=rl_risk_instruments.values(),
    instrument_labels=rl_risk_instruments.keys(),
    s=[r.rate().real for r in rl_risk_instruments.values()],
    id=live_curve.id(),
    func_tol=1e-8,
    conv_tol=1e-10,
)

SUCCESS: `func_tol` reached after 0 iterations (levenberg_marquardt), `f_val`: 0.0, `time`: 0.0191s


In [9]:
z5z6 = rl.Portfolio([
	# rl_ser("X25", risk=-50_000),
	# rl_ser("F26", risk=100_000),
	# rl_ser("G26", risk=-50_000),
    
	rl_fomc_swap(meeting="oct25", risk=-50_000),
    rl_fomc_swap(meeting="dec25", risk=100_000),
    rl_fomc_swap(meeting="jan26", risk=-50_000),
])

z5z6.delta(solver=rl_risk_solver).style.format("{:_.0f}")

# O/N Curve

In [10]:
x_data_num, y_data_rate = live_curve.handle()._plot_rates("1d", left=rl.NoInput(0), right=rl.NoInput(0))
plot_data_dict = dict(zip(x_data_num, [y.real for y in y_data_rate]))

calendar = ql.UnitedStates(ql.UnitedStates.FederalReserve)
filtered_plot_data = {dt: rate for dt, rate in plot_data_dict.items() if calendar.isBusinessDay(ql.Date(dt.day, dt.month, dt.year))}

x_business_days = list(filtered_plot_data.keys())
y_business_rates = list(filtered_plot_data.values())
curve_nodes = live_curve.handle().nodes._nodes.keys()

# fig, ax = plt.subplots()
# ax.plot(x_business_days, y_business_rates)
# ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
# plt.xticks(rotation=45, ha="right")  # Rotate ticks for better readability
# ax.grid(True, linestyle="--", alpha=0.6)

# ticks = [t.tz_localize(None) if getattr(t, "tzinfo", None) else t for t in curve_nodes]
# ax.set_xticks(mdates.date2num(ticks))
# ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
# plt.title(f"{curve_handle.meta()["id"]} | {curve_handle.meta()["timestamp"]} | 1d")
# plt.xticks(rotation=45)
# plt.tight_layout()
# plt.show()


fig = go.Figure()
fig.add_trace(go.Scatter(x=x_business_days, y=y_business_rates, mode="lines"))

tick_vals = [pd.Timestamp(t).tz_localize(None) for t in curve_nodes]
tick_text = [pd.Timestamp(t).strftime("%Y-%m-%d") for t in tick_vals]

fig.update_layout(
    title=f"{live_curve.meta()["id"]} | {live_curve	.meta()["timestamp"]} | 1d Curve",
    template="plotly_dark",
    margin=dict(l=40, r=20, t=60, b=80),
    xaxis=dict(tickmode="array", tickvals=tick_vals, ticktext=tick_text, tickangle=45, showgrid=True),
    height=600,
    yaxis=dict(showgrid=True),
)
fig.update_xaxes(
    showspikes=True,
    spikecolor="white",
    spikesnap="cursor",
    spikemode="across",
    showgrid=True,
)
fig.update_yaxes(
    showspikes=True,
    spikecolor="white",
    spikesnap="cursor",
    spikethickness=0.5,
    showgrid=True,
)

fig.show()

In [14]:
irssfm = IRSwapStructureFunctionMap(curve=live_curve)

spot_queries = [
    IRSwapQuery(curve=curve, tenor="1D", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),
    IRSwapQuery(curve=curve, tenor="1W", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),
    IRSwapQuery(curve=curve, tenor="2W", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),
    IRSwapQuery(curve=curve, tenor="3W", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),
    IRSwapQuery(curve=curve, tenor="1M", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),
    IRSwapQuery(curve=curve, tenor="2M", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),
    IRSwapQuery(curve=curve, tenor="3M", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),
    IRSwapQuery(curve=curve, tenor="4M", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),
    IRSwapQuery(curve=curve, tenor="5M", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),
    IRSwapQuery(curve=curve, tenor="6M", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),
    IRSwapQuery(curve=curve, tenor="7M", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),
    IRSwapQuery(curve=curve, tenor="8M", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),
    IRSwapQuery(curve=curve, tenor="9M", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),
    IRSwapQuery(curve=curve, tenor="10M", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),
    IRSwapQuery(curve=curve, tenor="11M", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),
    IRSwapQuery(curve=curve, tenor="1Y", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),
    IRSwapQuery(curve=curve, tenor="15M", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),
    IRSwapQuery(curve=curve, tenor="18M", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),
    IRSwapQuery(curve=curve, tenor="2Y", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),
]

rl_spots = []
for q in spot_queries:
    pkg, _ = irssfm.apply(q.structure, tenor=q.tenor, **q.structure_kwargs)
    rl_spots.append(pkg[0])

x = [live_curve.calendar_advance(live_curve.reference_date(), s.__dict__["kwargs"]["termination"]) for s in rl_spots]
y = [s.rate().real for s in rl_spots]

fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=y, mode="lines", name="1D"))
tick_vals = [pd.Timestamp(t).tz_localize(None) for t in x]
tick_text = [pd.Timestamp(t).strftime("%Y-%m-%d") for t in tick_vals]
fig.update_layout(
    title=f"{live_curve.meta()["id"]} | {live_curve.meta()["timestamp"]} | spot term curve",
    template="plotly_dark",
    margin=dict(l=40, r=20, t=60, b=80),
    xaxis=dict(tickmode="array", tickvals=tick_vals, ticktext=tick_text, tickangle=45, showgrid=True),
    height=700,
    yaxis=dict(showgrid=True),
)
fig.update_xaxes(
    showspikes=True,
    spikecolor="white",
    spikesnap="cursor",
    spikemode="across",
    showgrid=True,
)
fig.update_yaxes(
    showspikes=True,
    spikecolor="white",
    spikesnap="cursor",
    spikethickness=0.5,
    showgrid=True,
)
fig.show()